## Preamble and sampling


In [2]:
import nltk
import polars as pl

from collections import Counter
from datasets import load_dataset
from nltk.tokenize import word_tokenize

/home/ubuntu/miniconda/envs/py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:
nltk.download("punkt")
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /home/ubuntu/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/ubuntu/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [4]:
load_dataset("roneneldan/TinyStories", split="train").to_parquet("tinystories.parquet")
load_dataset("SimpleStories/SimpleStories", split="train").to_parquet("simplestories.parquet")

Creating parquet from Arrow format: 100%|██████████| 32/32 [01:47<00:00,  3.34s/ba]


3142783327

In [7]:
tinystories_df = pl.scan_parquet("tinystories.parquet")
tinystories_df.head().collect()

text
str
"""One day, a little girl named L…"
"""Once upon a time, there was a …"
"""One day, a little fish named F…"
"""Once upon a time, in a land fu…"
"""Once upon a time, there was a …"


In [8]:
simplestories_df = pl.scan_parquet("simplestories.parquet")
simplestories_df.head().collect()

story,topic,theme,style,feature,grammar,persona,initial_word_type,initial_letter,word_count,character_count,num_paragraphs,avg_word_length,avg_sentence_length,flesch_reading_ease,flesch_kincaid_grade,dale_chall_readability_score,num_stories_in_completion,expected_num_stories_in_completion,generation_id,model
str,str,str,str,str,str,str,str,str,i64,i64,i64,f64,f64,f64,f64,f64,i64,i64,str,str
"""Eagerly, a girl named Kim went…","""subterranean worlds""","""Hardship""","""lighthearted""","""symbolism""","""""","""""","""adverb""","""T""",109,570,2,4.34,15.57,74.49,6.3,8.26,11,12,"""7482221ff291ce5936eccc77c700d9…","""gpt-4o-mini-2024-07-18"""
"""Key turned in the old lock. A …","""shape-shifting""","""Failure""","""playful""","""symbolism""","""""","""""","""noun""","""K""",391,1769,6,3.7,10.29,96.28,2.0,7.14,6,6,"""940305166197525600cfc5fcbbb513…","""gpt-4o-mini-2024-07-18"""
"""Rain poured down on the tiny t…","""seasonal changes""","""Friendship""","""humorous""","""irony""","""perfect aspect""","""someone evil""","""noun""","""C""",510,2077,7,3.34,9.81,96.28,2.0,6.09,4,5,"""315219dcf08e93102350aebb412a2c…","""gpt-4o-mini-2024-07-18"""
"""An old tree stood tall in a fo…","""virtual worlds""","""Discovery""","""whimsical""","""circular narrative structure""","""""","""""","""adjective""","""O""",120,567,2,3.84,15.0,83.36,4.9,6.83,11,12,"""1a854e00b6750677b6856a0be0530e…","""gpt-4o-mini-2024-07-18"""
"""Truthfully, the garden was a p…","""gardens""","""Contradiction""","""suspenseful""","""climactic structure""","""""","""""","""adverb""","""T""",466,2148,8,3.77,11.1,87.31,3.4,7.13,4,4,"""5f5524c5a3ac334bdb6c576e531304…","""gpt-4o-mini-2024-07-18"""


## Length comparison

In [9]:
tinystories_df = tinystories_df.with_columns(
    pl.col("text").str.split(" ").list.len().alias("word_count")
)
tinystories_df.head().collect()

text,word_count
str,u32
"""One day, a little girl named L…",132
"""Once upon a time, there was a …",140
"""One day, a little fish named F…",166
"""Once upon a time, in a land fu…",161
"""Once upon a time, there was a …",127


In [30]:
from readability import Readability

text

NameError: name 'text' is not defined

In [28]:
from readability import Readability

def calculate_flesch_kincaid_score(text):
    r = Readability(text)
    result = r.flesch_kincaid(min_words=10)
    return result.score

calculate_flesch_kincaid_score(tinystories_df.head().collect().item(1, "text"))

tinystories_df.with_columns(
    pl.col("text").map_elements(calculate_flesch_kincaid_score, return_dtype=pl.Float64)
        .alias("flesch_kincaid_score")
).collect()
    

TypeError: Readability.flesch_kincaid() got an unexpected keyword argument 'min_words'

In [14]:
ts_result = tinystories_df.select(
    word_count_mean = pl.col("word_count").mean(),
    word_count_std = pl.col("word_count").std(),
    
)
ts_result.collect()

word_count_mean,word_count_std
f64,f64
171.832831,77.416249


In [15]:
ts_result